In [5]:
# 환경 설정 및 라이브러리 설치
!pip install -q openai langchain langchain-openai langchain-community faiss-cpu \
    rank_bm25 pandas numpy matplotlib gradio python-dotenv tiktoken

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 462.4/462.4 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 21.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unstructured-client 0.44.0 requires aiofiles>=25.1.0, but you have aiofiles 24.1.0 which is incompatible.
unstructured-client 0.44.0 requires pydantic>=2.12.5, but you have pydantic 2.12.3 which is incompatible.


In [6]:
import os
import json
import time
import random
from pathlib import Path
# from dotenv import load_dotenv
# load_dotenv()
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from openai import OpenAI
oai = OpenAI()
llm = ChatOpenAI(model="gpt-4o-mini")

In [7]:
!pip install -q "unstructured[pdf,docx,pptx]" unstructured-inference
# !pip install -q langchain langchain-community langchain-openai langchain-text-splitters
# !pip install -q faiss-cpu tiktoken python-dotenv
# !pip install -q pandas tabulate

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires aiofiles<25.0,>=22.0, but you have aiofiles 25.1.0 which is incompatible.
gradio 5.50.0 requires pydantic<=2.12.3,>=2.0, but you have pydantic 2.13.4 which is incompatible.


# 10주차 3-4차시: 문서 처리 파이프라인 (PDF 파싱, 청킹, 테이블 추출)

| 주제 | 내용 |
|---|---|
| 문서 파싱 | PDF 구조 분석, 메타데이터 추출 |
| 청킹 | 문서 분할 전략 |
| 텍스트 활용 | 파싱된 텍스트 전처리 |
| 배치 처리 | 클래스 기반 PDF 일괄 처리 |
| 테이블 추출 | PDF → Markdown 테이블 변환, 정제 |

In [8]:
import os
import warnings
warnings.filterwarnings('ignore')
from dotenv import load_dotenv
load_dotenv()
SAMPLE_DIR = "/content/drive/MyDrive/Colab Notebooks/2026-05-21 LLM 교육과정 실습/samples"
BATCH_DIR = os.path.join(SAMPLE_DIR, "batch")

In [9]:
from unstructured.partition.auto import partition

In [10]:
pdf_path = os.path.join(SAMPLE_DIR, 'business_report.pdf')

In [11]:
pdf_path

'/content/drive/MyDrive/Colab Notebooks/2026-05-21 LLM 교육과정 실습/samples/business_report.pdf'

In [12]:
elements = partition(filename = pdf_path)

In [13]:
for i, el in enumerate(elements[:5]):
  print(el.category, "|||", el)

Title ||| 2024 Annual Business Report
Title ||| TechCorp International Inc.
Title ||| Executive Summary
NarrativeText ||| This report presents TechCorp International's financial performance for fiscal year 2024. The company achieved record revenue of $2.8 billion, representing a 23% year-over-year growth. Our AI and cloud services division drove the majority of this growth, contributing 65% of total revenue.
NarrativeText ||| Key strategic initiatives including the acquisition of DataFlow Systems and expansion into the Asia-Pacific market have positioned the company for sustained growth in 2025 and beyond.


In [14]:
from unstructured.partition.auto import partition
from unstructured.partition.pdf import partition_pdf


In [15]:
elements = partition_pdf(filename = pdf_path, strategy = 'fast')

In [16]:
e = elements[0]

In [17]:
type(e)

unstructured.documents.elements.Title

In [18]:
e.category, e.id

('Title', '70aab7bb68d429fd46386e78fdc8bba1')

In [19]:
e.metadata.page_number, e.metadata.filename, e.metadata.filetype

(1, 'business_report.pdf', 'application/pdf')

In [20]:
# fast : text only
# hi-res : 레이아웃, 테이블, 이미지 등을 같이 확인
  # layout(테이블 + 이미지)을 볼 때, detectron, yolox 등을 이용해서 처리

In [21]:
# hi-res 한번 확인

elements_hi = partition_pdf(filename=pdf_path, strategy = 'hi_res', infer_table_structure=True)

yolox_l0.05.onnx:   0%|          | 0.00/217M [00:00<?, ?B/s]

TypeError: DecoratorInfos.build() got an unexpected keyword argument 'replace_wrapped_methods'

In [ ]:
for c, n in Counter(el.category for el in elements).most_common():
  print(f" {c} : {n}")

In [ ]:
from collections import Counter

In [ ]:
for c, n in Counter(el.category for el in elements_hi).most_common():
  print(f" {c} : {n}")

In [ ]:
# 실습

In [ ]:
def analyze_document(pdf_path, strategy="fast"):
    elements = partition_pdf(filename=pdf_path, strategy=strategy)

    return {
        "total_elements": len(elements),
        "type_statistics": dict(Counter(el.category for el in elements)),
        "top_body": sorted([{
                    "length": len(el.text),
                    "page": el.metadata.page_number,
                    "text_preview": el.text[:20]
                }
                for el in elements
                if el.category == "NarrativeText"
            ],
            key=lambda x: x["length"],
            reverse=True
        )[:5]
    }

In [ ]:
analyze_document(pdf_path)

In [ ]:
# 강사님 코드
def analyze_document(pdf_path: str, strategy: str = "fast") -> dict:
    els = partition_pdf(filename=pdf_path, strategy=strategy)
    type_stats = dict(Counter(el.category for el in els))
    page_dist = defaultdict(lambda: Counter())
    for el in els:
        p = int(el.metadata.page_number or 0)
        page_dist[p][el.category] += 1
    narratives = [el for el in els if el.category == "NarrativeText"]
    narratives.sort(key=lambda e: len(str(e)), reverse=True)
    top3 = [
        {
            "length": len(str(el)),
            "page": int(el.metadata.page_number or 0),
            "text_preview": str(el)[:80],
        }
        for el in narratives[:3]]
    return {
        "total_elements": len(els),
        "type_statistics": type_stats,
        "page_distribution": {p: dict(c) for p, c in page_dist.items()},
        "top3_longest_narratives": top3,
    }

### 메타데이터
- 페이지 넘버나, 파일타입, 각주 정보 등
  - coordinates : 어떤 좌표에 있는지 알 수 있음
  - 좌표 -> title, text, table, image 등
- 문서의 hierachy structure 등, 구조화된 정보 얻을 수 있음

- pageindex: 올해 나온 라이브러리
  - 링크: https://github.com/VectifyAI/PageIndex

In [ ]:
meta_dict = elements[0].metadata.to_dict()

In [ ]:
meta_dict

In [ ]:
from unstructured.partition.html import partition_html
from unstructured.partition.docx import partition_docx
from unstructured.partition.pptx import partition_pptx

In [ ]:
html_path = os.path.join(SAMPLE_DIR, 'product_page.html')
docx_path = os.path.join(SAMPLE_DIR, 'quarterly_report.docx')
pptx_path = os.path.join(SAMPLE_DIR, 'investor_deck.pptx')

In [ ]:
html_els = partition_html(filename = html_path)
docx_els = partition_docx(filename = docx_path)
pptx_els = partition_pptx(filename = pptx_path)

In [ ]:
len(html_els), len(docx_els), len(pptx_els)

In [ ]:
import pandas as pd

rows = []

for fmt, els in [('html', html_els), ('docx', docx_els), ('pptx', pptx_els)]:
  cats = Counter(el.category for el in els)
  rows.append({'format':fmt, 'total':len(els), **dict(cats)})

In [ ]:
df = pd.DataFrame(rows).fillna(0).set_index('format')

In [ ]:
print(df.to_string())

In [ ]:
# from unstructured.partition.text import partition_text # 메모장 등의 형식

# class MultiFormatParser: # 실제 사용하려면 parser를 달아주면 됨
#   PARSERS = {
#       '.pdf': partition_pdf,
#       '.html' : partition_html,
#       '.txt' : partition_text
#   }

#   def __init__(self):
#     self.results = {}

#   def parse(self, path:str) -> list:
#     ext = os.path.splitext(path)[1].lower() # extension
#     els = list(self.PARSERS[ext](filename=path))
#     self.results[path] = els
#     return els

## 청킹 (Chunking)


In [ ]:
from unstructured.chunking.title import chunk_by_title # 제목 기준으로 분할
from unstructured.chunking.basic import chunk_elements # 텍스트 splitter(랭체인에서 했었던), 청크 사이즈 및 overlap 크기에 따라 청킹

In [ ]:
# 문자열 개수, 토큰 개수로 하기도 했으나,
# recursive text splitter 등을 이용하면 문단 페이지 단위로, 청크사이즈보다 크면 문단 단위로, 작으면 문장 단위로 등등

In [ ]:
chunks_title = chunk_by_title(
    elements,
    max_characters = 1000,
    new_after_n_chars = 800,
    combine_text_under_n_chars = 200, # 200자 이하는 기존 청크에 합쳐라.
)

In [ ]:
chunk_basic = chunk_elements(elements, max_characters=500, overlap=50)

In [ ]:
chunks_title

In [ ]:
str(chunks_title[0])[:200]

In [ ]:
elements = partition_pdf(filename = pdf_path, strategy = 'fast')

In [ ]:
len(elements)

In [ ]:
avg_len = sum(len(str(el))for el in elements) / len(elements)

In [ ]:
avg_len

In [ ]:
# 청킹을 할 때, 청크 사이즈 옵션만 주고 할 수도 있으나,
# 문서 특성에 따라서는 chunk_by_title을 쓰는데
# max_characters를 능동적으로 조절하면 좋겠음
# 문서들의 길이를 봐서 평균 텍스트의 길이가..

def adaptive_chunk(elements: list) -> list:
  # elements의 평균 텍스트 길이
  text_lengths = [len(str(el)) for el in elements if str(el).strip()]
  avg_len = sum(text_lengths) / len(text_lengths) if text_lengths else 0

  # 평균 text 길이에 따라 max_characters 설정
  if avg_len >= 200:
      max_characters = 1000
      new_after_n_chars = 800
  elif avg_len >= 50:
      max_characters = 500
      new_after_n_chars = 400
  else:
      max_characters = 300
      new_after_n_chars = 250

  combine_text_under_n_chars = 200

  chunks_title = chunk_by_title(
      elements,
      max_characters=max_characters,
      new_after_n_chars=new_after_n_chars,
      combine_text_under_n_chars=combine_text_under_n_chars,
  )

  return chunks_title

In [ ]:
def adaptive_chunk(elements: list) -> list:
    # elements의 평균 텍스트 길이
    text_lengths = [len(str(el)) for el in elements if str(el).strip()]
    avg_len = sum(text_lengths) / len(text_lengths) if text_lengths else 0

    # 평균 text 길이에 따라 max_characters 설정
    if avg_len >= 200:
        max_characters = 1000
        new_after_n_chars = 800
    elif avg_len >= 50:
        max_characters = 500
        new_after_n_chars = 400
    else:
        max_characters = 300
        new_after_n_chars = 250

    combine_text_under_n_chars = 200

    chunks_title = chunk_by_title(
        elements,
        max_characters=max_characters,
        new_after_n_chars=new_after_n_chars,
        combine_text_under_n_chars=combine_text_under_n_chars,
    )

    return chunks_title

In [ ]:
# 텍스트 정제 관련 라이브러리
from unstructured.cleaners.core import clean, clean_extra_whitespace, replace_unicode_quotes
# 띄어쓰기,

In [ ]:
samples = [
    "  This   is   a    test   with   extra   spaces.  ",
    "Revenue was “$2.8 billion” in FY2024.",
    "Line\n\n\n\nwith too many\nlinebreaks",
]

In [ ]:
for s in samples:
  cleaned = clean_extra_whitespace(replace_unicode_quotes(s))
  print(cleaned)

## 랭체인에 추출 및 처리된 텍스트 활용하기

In [ ]:
from langchain_community.document_loaders import UnstructuredFileLoader

In [ ]:
loader = UnstructuredFileLoader(pdf_path, mode = 'elements', strategy = 'fast')

In [ ]:
documents = loader.load()

In [ ]:
documents # 랭체인 로더로 불러오는 경우, 랭체인에서 바로 활용할 수 있도록 Document형식, page_content 등에 담아주는 것을 확인할 수 있음
# coordinates, layout_height 등의 속성도 확인가능

In [ ]:
documents[0].page_content

In [ ]:
documents[0].metadata.items()

In [ ]:
for mode in ['single', 'elements', 'paged']:
  loader = UnstructuredFileLoader(pdf_path, mode = mode, strategy = 'fast')
  docs = loader.load()
  avg = sum(len(d.page_content) for d in docs) / max(len(docs), 1)
  print(f"{mode} {len(docs)} {avg} {docs[0].page_content[:30]}")

In [ ]:
def answer_from_pdf(pdf_path, question, k=3) -> dict:

  1) pdf_path -> docs
  2) embeddings = OpenAIEmbedddings(model = 'text-embedding-3-small')
  vectorstrore = FAISS.from_documents(docs, embeddings)

  3) retrieve 하고
  4) generate 까지 -> llm.invoke() 안에 SystemMessage(content = '주어진 컨텍스트에 근거해 답변하세요'), HumanMessage(content= "컨텍스트-> retrieve \n질문{question})

  return {
      'answer' : msg.content,
      'source': 어떤 page인지, preview : page.content[:30]
  }

In [ ]:
from langchain_community.document_loaders import UnstructuredPDFLoader
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.messages import SystemMessage, HumanMessage

In [ ]:
def answer_from_pdf(pdf_path, question, k=3) -> dict:
    # 1) pdf_path -> docs

    loader = UnstructuredFileLoader(pdf_path, mode = 'elements', strategy = 'fast')
    documents = loader.load()

    # 2) embeddings -> vectorstore
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    vectorstore = FAISS.from_documents(documents, embeddings)

    # 3) retrieve
    retriever = vectorstore.as_retriever(search_kwargs={"k": k})
    retrieved_docs = retriever.invoke(question)

    context = "\n\n".join([f"[문서{i+1}]\n{doc.page_content}" for i, doc in enumerate(retrieved_docs)])

    # 4) generate
    llm = ChatOpenAI(model="gpt-4o-mini")

    msg = llm.invoke([
        SystemMessage(content="주어진 컨텍스트에 근거해 답변하세요."),
        HumanMessage(content=f"컨텍스트:\n{context}\n\n질문: {question}")
    ])

    return {
        "answer": msg.content,
        "source": [
            {
                "page": doc.metadata.get("page_number", doc.metadata.get("page", None)),
                "preview": doc.page_content[:30]
            }
            for doc in retrieved_docs
        ]
    }

In [ ]:
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [ ]:
question = "이 문서의 내용은 뭐니?"
answer_from_pdf(pdf_path, question)

In [ ]:
# 강사님 코드

def answer_from_pdf(pdf_path, question, k=3) -> dict:
    # 1) pdf_path -> docs

    els = partition_pdf(filename=pdf_path, strategy='fast')
    cleaned = [el for el in els if el.category not in ['Header', 'Footer', 'PageBreak']]
    chunks = chunk_by_title(
        cleaned,
        max_characters = 600,
        combine_text_under_n_chars = 150,
    )

    docs = [
        Document(
            page_content = str(c),
            metadata = {
                'source' : os.path.basename(pdf_path),
                'page' : getattr(c.metadata, 'page_number', None),
                'category': getattr(c.metadata, 'category', None)
            }
        ) for c in chunks if str(c).strip()
    ]

    # 2) embeddings -> vectorstore
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    vectorstore = FAISS.from_documents(documents, embeddings)

    # 3) retrieve
    hits = vectorstore.similarity_search(question, k=k)
    context = "\n\n".join(f"[p{h.metadata.get('page','?')}] {h.page_content}" for h in hits)

    # 4) generate
    llm = ChatOpenAI(model="gpt-4o-mini")
    msg = llm.invoke([
        SystemMessage(content="주어진 컨텍스트에 근거해 답변하세요."),
        HumanMessage(content=f"컨텍스트:\n{context}\n\n질문: {question}")
    ])

    return {
        "answer": msg.content,
        "source": [
            {
                "page": doc.metadata.get("page_number", doc.metadata.get("page", None)),
                "preview": doc.page_content[:30]
            }
            for doc in retrieved_docs
        ]
    }

## 4차시 (배치 처리 & 테이블 추출)

### 배치 처리

In [35]:
import glob
batch_files = sorted(glob.glob(os.path.join(BATCH_DIR, '*pdf')))

In [36]:
batch_files

['/content/drive/MyDrive/Colab Notebooks/2026-05-21 LLM 교육과정 실습/samples/batch/report_engineering.pdf',
 '/content/drive/MyDrive/Colab Notebooks/2026-05-21 LLM 교육과정 실습/samples/batch/report_finance.pdf',
 '/content/drive/MyDrive/Colab Notebooks/2026-05-21 LLM 교육과정 실습/samples/batch/report_hr.pdf',
 '/content/drive/MyDrive/Colab Notebooks/2026-05-21 LLM 교육과정 실습/samples/batch/report_legal.pdf',
 '/content/drive/MyDrive/Colab Notebooks/2026-05-21 LLM 교육과정 실습/samples/batch/report_marketing.pdf',
 '/content/drive/MyDrive/Colab Notebooks/2026-05-21 LLM 교육과정 실습/samples/batch/report_operations.pdf',
 '/content/drive/MyDrive/Colab Notebooks/2026-05-21 LLM 교육과정 실습/samples/batch/report_product.pdf',
 '/content/drive/MyDrive/Colab Notebooks/2026-05-21 LLM 교육과정 실습/samples/batch/report_research.pdf',
 '/content/drive/MyDrive/Colab Notebooks/2026-05-21 LLM 교육과정 실습/samples/batch/report_sales.pdf',
 '/content/drive/M

#### 클래스 설명
- PDF 파일 여러 개를 배치 처리하면서, 이미 처리한 파일은 체크포인트로 건너뛰는 클래스
- ckpt_path에 저장된 처리 완료 파일 목록을 불러옴
- paths에 있는 PDF들을 하나씩 순회
- 이미 완료된 파일이면 SKIP
- 새 파일이면:
  - partition_pdf()로 PDF를 element 단위로 분해
  - chunk_by_title()로 chunk 생성
  - 파일명, element 개수, chunk 개수 저장
  - 처리 시간과 성공 상태 기록
  - 완료 목록에 파일명 추가
  - 체크포인트 파일에 저장

In [45]:
import os, json, time

class BatchProcessor:
    def __init__(self, ckpt_path):
        self.ckpt_path = ckpt_path
        self.completed = self.__load_ckpt() # 체크포인트로 가져온 값을 completed 변수에 넣음
        self.results = []

    def __load_ckpt(self):
        if os.path.exists(self.ckpt_path):
            with open(self.ckpt_path, "r", encoding="utf-8") as f:
                return set(json.load(f))
        return set()

    def __save_ckpt(self):
        with open(self.ckpt_path, "w", encoding="utf-8") as f:
            json.dump(list(self.completed), f, ensure_ascii=False, indent=2)

    def process_one(self, path):
        els = partition_pdf(filename=path, strategy="fast")
        chunks = chunk_by_title(els, max_characters=600)

        return { # 어떤 파일에서 왔고, element의 개수(타이틀, 리스트 ...등등), chunk는 말뭉치의 개수
            "file": os.path.basename(path),
            "n_elements": len(els),
            "n_chunks": len(chunks)
        }

    def run(self, paths):
        for p in paths: # paths에 있는 애들을 for 루프로 돌면서
            name = os.path.basename(p)

            if name in self.completed:
                print(f"SKIP (done): {name}")
                continue # 다음 루프로 넘어감

            try:
                t0 = time.time()
                r = self.process_one(p)
                r["time"] = round(time.time() - t0, 2)
                r["status"] = "success"

                self.results.append(r)
                self.completed.add(name)
                self.__save_ckpt()

                print(f"OK: {name} ({r['n_chunks']} chunks, {r['time']}s)")

            except Exception as e:
                self.results.append({
                    "file": name,
                    "status": "failed",
                    "error": str(e)
                })
                print(f"FAIL: {name} ({e})")

        return self.results

In [46]:
ckpt = '260522_ckpt.json'
bp = BatchProcessor(ckpt)

In [47]:
result = bp.run(batch_files[:5])

FAIL: report_engineering.pdf (name 'chunk_by_title' is not defined)


FAIL: report_finance.pdf (name 'chunk_by_title' is not defined)


FAIL: report_hr.pdf (name 'chunk_by_title' is not defined)


FAIL: report_legal.pdf (name 'chunk_by_title' is not defined)


FAIL: report_marketing.pdf (name 'chunk_by_title' is not defined)


In [48]:
result

[{'file': 'report_engineering.pdf',
  'status': 'failed',
  'error': "name 'chunk_by_title' is not defined"},
 {'file': 'report_finance.pdf',
  'status': 'failed',
  'error': "name 'chunk_by_title' is not defined"},
 {'file': 'report_hr.pdf',
  'status': 'failed',
  'error': "name 'chunk_by_title' is not defined"},
 {'file': 'report_legal.pdf',
  'status': 'failed',
  'error': "name 'chunk_by_title' is not defined"},
 {'file': 'report_marketing.pdf',
  'status': 'failed',
  'error': "name 'chunk_by_title' is not defined"}]

In [34]:
!pip install -q "unstructured[pdf,docx]" pymupdf camelot-py[base] Pillow matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 69.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 2.0 MB/s eta 0:00:00


In [39]:
import os
import io
import re
import json
import base64
import warnings
from pathlib import Path
from typing import List, Optional
import pandas as pd
import fitz  # PyMuPDF (설치명인데, 라이브러리 이름은 fitz)
from PIL import Image
from dotenv import load_dotenv
load_dotenv()
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
llm        = ChatOpenAI(model="gpt-4o-mini", temperature=0)
llm_vision = ChatOpenAI(model="gpt-4o-mini", temperature=0, max_tokens=600)
SAMPLE_DIR = "/content/drive/MyDrive/Colab Notebooks/2026-05-21 LLM 교육과정 실습/samples"
BATCH_DIR  = os.path.join(SAMPLE_DIR, "batch")
PDF_BUSINESS = os.path.join(SAMPLE_DIR, "business_report.pdf")
PDF_BENCH    = os.path.join(SAMPLE_DIR, "benchmark_test.pdf")
DOCX_REPORT  = os.path.join(SAMPLE_DIR, "quarterly_report.docx")

#### PDF에서 테이블 추출 후 Markdown 변환 or RAG 활용

In [49]:
# PDF는 텍스트로 구성이된 벡터타입 포맷
  # pymupdf camelot 등은 텍스트 좌표, 가로선 등을 이용해서 테이블을 추측 (패턴을 활용)
# 다음주는 멀티모달, 비전모델 등을 이용해 탐색

In [50]:
doc = fitz.open(PDF_BUSINESS)

In [51]:
page = doc[0]

In [52]:
len(doc)

3

In [53]:
page.rect.width

595.2755737304688

In [54]:
len(page.get_text('blocks'))

12

In [56]:
for b in page.get_text('blocks'):
  x0, y0, x1, y1, txt = b[0], b[1], b[2], b[3], b[4]
  print(f" ({x0}, {y0} - ({x1}, {y1}) | {txt[:10]}")

 (128.25779724121094, 61.01287078857422 - (467.0177917480469, 94.06086730957031) | 2024 Annua
 (213.21080017089844, 113.64286804199219 - (382.06475830078125, 132.87887573242188) | TechCorp I
 (78.0, 165.5728759765625 - (230.06399536132812, 187.60487365722656) | Executive 
 (78.0, 199.8678741455078 - (512.6869506835938, 262.98187255859375) | This repor
 (78.0, 271.86785888671875 - (512.0709838867188, 318.98187255859375) | Key strate
 (78.0, 361.7828674316406 - (201.5260009765625, 379.6838684082031) | Financial 
 (156.46871948242188, 395.94287109375 - (466.0450744628906, 409.682861328125) | Metric
202
 (139.2337188720703, 423.94287109375 - (452.29510498046875, 437.682861328125) | Revenue ($
 (132.8487091064453, 451.94287109375 - (452.29510498046875, 465.682861328125) | Net Income
 (131.45370483398438, 479.94287109375 - (454.8000793457031, 493.682861328125) | Operating 
 (125.34371185302734, 507.94287109375 - (452.29510498046875, 521.682861328125) | R&D Spendi
 (145.3487091064453, 535.942

In [ ]:
# 원래는 박스 영역인데
# CV의 경우 중심점 및 width, height를 이용하는 경우도 있음

# normalize 하는 경우도 있음

#### 테이블 추출

In [57]:
all_tables = []
for page_idx, page in enumerate(doc):
  tables = page.find_tables() # 테이블 추출
  for t_idx, table in enumerate(tables.tables):
    df = table.to_pandas()
    all_tables.append({'page': page_idx +1, 'shape': df.shape, 'df': df})

Consider using the pymupdf_layout package for a greatly improved page layout analysis.


In [58]:
len(all_tables)

2

In [60]:
for t in all_tables:
  print(f"\n Page {t['page']} | shape = {t['shape']}")
  print(t['df'].head(3).to_string())


 Page 1 | shape = (5, 4)
             Metric   2024   2023 Change (%)
0      Revenue ($B)   2.80   2.28       +23%
1   Net Income ($M)    420    310       +35%
2  Operating Margin  18.5%  15.2%     +3.3pp

 Page 2 | shape = (4, 5)
          Segment Revenue ($M) Growth Margin Customers
0      AI & Cloud        1,820   +31%    22%      500+
1   Enterprise SW          700   +12%    35%    2,100+
2  Prof. Services          280    +8%    25%      800+


In [63]:
def analyze_tables(pdf_path):
  # pdf 경로 받아서, 딕셔너리 형태로 리턴
  # 테이블 개수
  doc = fitz.open(pdf_path)
  all_tables = []

  for page_idx, page in enumerate(doc):
    tables = page.find_tables() # 테이블 추출
    for t_idx, table in enumerate(tables.tables):
      df = table.to_pandas()
      all_tables.append({
          'page': page_idx +1, 'shape': df.shape,
          "table_idx": t_idx + 1,
          "rows": df.shape[0], "cols": df.shape[1],
          "columns": list(df.columns),'df': df
      })
  return {"n_tables": len(all_tables),"tables": [{ "page": t["page"],
                                                  "table_idx": t["table_idx"],
                                                   "rows": t["rows"],
                                                   "cols": t["cols"],
                                                   "columns": t["columns"]
                                                   } for t in all_tables],
        "total_cells": sum(t["rows"] * t["cols"] for t in all_tables),
    }


In [64]:
analyze_tables(PDF_BENCH)

{'n_tables': 3,
 'tables': [{'page': 1,
   'table_idx': 1,
   'rows': 3,
   'cols': 5,
   'columns': ['Category', 'Q1', 'Q2', 'Q3', 'Q4']},
  {'page': 1,
   'table_idx': 2,
   'rows': 3,
   'cols': 5,
   'columns': ['Category', 'Q1', 'Q2', 'Q3', 'Q4']},
  {'page': 1,
   'table_idx': 3,
   'rows': 3,
   'cols': 5,
   'columns': ['Category', 'Q1', 'Q2', 'Q3', 'Q4']}],
 'total_cells': 45}

In [ ]:
# unstructured, partition 등으로 파싱했었음
  # hires 기준 -> yolox, detecctron2 등을 사용 (느림)
    # 객체 탐지 기반 비전모델임
# pymupdf도 썼었음 -> 가장 쉽게 사용가능 (다른 라이브러리에 의존성이 없음)
  # 복잡한 표에서는 정확도가 떨어짐
# 마지막으로 camelot이 있는데..
  # pymupdf보다는 정확도 높게 표를 가져옴
  # 선이 명확한 표에 대한 파싱 정확도가 높음


In [66]:
!apt-get update -qq
!apt-get install -y ghostscript
# camelot의 flavor="lattice"는 내부적으로 PDF를 이미지로 변환해야 해서 Ghostscript가 필요

Processing triggers for man-db (2.10.2-1) ...


In [67]:
import camelot

cam_tables = camelot.read_pdf(PDF_BUSINESS, pages = 'all', flavor = 'lattice') # lattice는 격자

In [71]:
for i, t in enumerate(cam_tables):
  print(f"Table {i+1} | page = {t.page} | shape = {t.df.shape} | accuracy = {t.accuracy}")
  print(t.df.head(2).to_string())

Table 1 | page = 1 | shape = (6, 4) | accuracy = 100.00000000000003
                                0     1     2     3
0  Metric\n2024\n2023\nChange (%)                  
1                    Revenue ($B)  2.80  2.28  +23%
Table 2 | page = 2 | shape = (5, 5) | accuracy = 100.0
                                                  0      1     2    3     4
0  Segment\nRevenue ($M)\nGrowth\nMargin\nCustomers                        
1                                        AI & Cloud  1,820  +31%  22%  500+


In [105]:
doc = fitz.open(PDF_BUSINESS)
raw_df = doc[0].find_tables().tables[0].to_pandas()

In [72]:
els = partition_pdf(filename = PDF_BUSINESS, strategy = 'hi_res')
unst_tables = [el for el in els if el.category == 'Table']
for tb in unst_tables:
  print(f"\n page = {tb.metadata.page_number}")
  print(str(tb)[:50])


 page = 1
Metric 2024 2023 Change (%) Revenue ($B) 2.80 2.28

 page = 2
Segment Revenue ($M) Growth Margin Customers AI & 


#### 정제하는 함수 작성
- 영어/한국어 혼재 등
- 1) 첫번째 행이 헤더로 보이면 -> 컬럼명으로 올려주기
- 2) 결측값, N/A, -, "" -> 통일
- 3) 숫자처럼 보이는 열은 numeric으로 처리

In [74]:
def clean_table(df) -> pd.DataFrame:
  df = df.copy()
  if df.iloc[0].apply(lambda v: isinstance(v, str) and len(v<30).all()):
    df.columns = df.iloc[0]
    df = df.iloc[1:].reset_index(drop=True)

  df = df.replace({'N/A': None, '-': None, '': None})

  for col in df.columns:
    sample = df[col].dropna().astype(str) # dropna하면 결측치 제거됨. column명은 string인 애들이 뽑힘
    if sample.empty: # 샘플이 없다면
      continue # 다음 컬럼으로 넘어감
    if sample.str.match(r"^[\$\+\-]?[\d,]+\.?\d*%?$").mean() > 0.6: # 더하기+, 빼기-, 달러$, 등이 함께 나오면 숫자일 확률이 높으므로
      cleaned = df[col].astype(str).str.replace(r"[\$,%]", "", regex = True)
      df[col] = pd.to_numeric(cleaned)
  return df


In [106]:
raw_df

,Metric,2024,2023,Change (%)
0,Revenue ($B),2.80,2.28,+23%
1,Net Income ($M),420,310,+35%
2,Operating Margin,18.5%,15.2%,+3.3pp
3,R&D Spending ($M),580,490,+18%
4,Employees,"12,500","10,200",+23%


In [94]:
raw_df

,Segment,Revenue ($M),Growth,Margin,Customers
0,AI & Cloud,"1,820",+31%,22%,500+
1,Enterprise SW,700,+12%,35%,"2,100+"
2,Prof. Services,280,+8%,25%,800+
3,Total,"2,800",+23%,18.5%,"3,400+"


In [109]:
clean_df = clean_table(raw_df)

TypeError: '<' not supported between instances of 'str' and 'int'

In [110]:
def table_to_markdown(df : pd.DataFrame, max_rows: int = 10) -> str:

  # 그냥  | 셀내용 | 등 '|'로 가둬주면 되고
  # 제목 행은 |---| 등으로 구분해주면됨

  df = df.fillna("")
  header = "| " + " | ".join(str(c) for c in df.columns) + " |"
  sep = "| " + " | ".join(["---"]*len(df.columns)) + " |"
  rows = df.head(max_rows).astype(str).apply(
      lambda r: "| " + " | ".join(r) + " |", axis=1).tolist()
  out = '\n'.join([header, sep] + rows)

  if len(df) > max_rows:
    out += f"\n ...({len(df) - max_rows} more rows)"
  return out

In [111]:
md_text = table_to_markdown(clean_df)
print(md_text)

NameError: name 'clean_df' is not defined

In [114]:
 def analyze_image(image_path, prompt = '이 이미지에 보이는 모든 텍스트를 추출하고 한 문장으로 요약해줘'):
  with open(image_path , 'rb') as f:
      b64 = base64.b64encode(f.read()).decode('utf-8')
  ext = os.path.splitext(image_path)[1].lstrip('.').lower()
  if ext == 'jpg':
    ext = 'jpeg'

  msg = HumanMessage(content = [ # 지금까지는 content에 텍스트만 넣어놨었는데, 다양한 형태의 데이터를 넣을떄는 타입을 명시해줌
      {'type': 'text', 'text':prompt},
      {'type': 'image_url', 'image_url' : {'url': f'data:image/{ext}:base64,{b64}'}}
  ])

  return llm_vision.invoke([msg]).content


In [ ]:


img1 = SAMPLE_DIR + 'p1_img1.png'

imgs = []

In [115]:
doc = fitz.open(PDF_BUSINESS)
saved = []
for page_idx, page in enumerate(doc):
  for img_ix, img in enumerate(page.get_images(full=True)):
    xref = img[0]
    base_img = doc.extract_image[xref]

#### 파이프라인 구성
- pdf 이미지, 테이블, 등등 이제 묶어서 RAG에 활용할 예정

In [118]:
from langchain_core.documents import Document

class VisionOCR:
  def __init__(self, llm):
    self.llm = llm
    self.results = []

  def process(self, paths: list, prompt : str = "이미지에서 모든 내용을 추출하세요") -> list:
    self.results = []
    for p in paths:
      try:
        with open(p, 'rb') as f:
          b64 = base64.b64encode(f.read()).decode('utf-8')
          ext = os.path.splitext(p)[1].lstrip('.').lower()
          if ext == 'jpg':
            ext = 'jpeg'

          msg = HumanMessage(content = [ # 지금까지는 content에 텍스트만 넣어놨었는데, 다양한 형태의 데이터를 넣을떄는 타입을 명시해줌
              {'type': 'text', 'text':prompt},
              {'type': 'image_url', 'image_url' : {'url': f'data:image/{ext}:base64,{b64}'}}
          ])

          self.results.append({'path': p, 'text' : self.llm.invoke([msg]).content, 'error': None})
      except Exception as e:
        self.results.append({'path' : p, 'text': '', 'error': str(e)})

  def to_documents(self) -> list:
    # # self.results에 쌓이면 for loop을 돌면서, 에러가 없는 경우에만
    # docs = []

    # for result in self.results:
    #   if result['error'] is None:
    #     docs.append(
    #         Document(
    #             page_content = result['text'],
    #             metadata = {
    #                 "source": result['path']
    #             }
    #         )
    #     )
    # return docs

    return [Document(page_content = r['text'], metadata = {'path' : r['path']}) for r in self.results if r['text']]

In [ ]:
ocr = VisionOCR()